# 기업 재무·공시 이상징후 탐정 — 통합 분석
2026-09-28 로컬 Open DART 삼성전자(00126380), 2023·2024. SQL·Pandas를 같은 원본·키·기간으로 대조한다. 후보는 위험 확정이 아니다. 저장소 루트에서 `jupyter nbconvert --execute notebooks/analysis.ipynb --to notebook --output analysis.executed.ipynb`로 재실행한다.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
from src.finance_pandas_anjihyeong import analyze as finance_analyze
from src.disclosure_pandas_leejeongsu import analyze as disclosure_analyze
from src.validation_leejeongsu import compare_results
root = Path.cwd()
if not (root / 'sql/queries.sql').is_file():
    root = root.parent
db_path = root / 'data/processed/dart.sqlite'
if not db_path.is_file():
    raise FileNotFoundError('먼저 python -m src.build_db 실행: ' + str(db_path))
db = sqlite3.connect(db_path)
db.executescript((root / 'sql/queries.sql').read_text(encoding='utf-8'))
finance = pd.read_sql_query('SELECT * FROM finance', db)
disclosures = pd.read_sql_query('SELECT * FROM disclosures', db)
print('원본', {'finance': len(finance), 'disclosures': len(disclosures)})

C:\Users\USER\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


원본 {'finance': 60, 'disclosures': 438}


In [2]:
finance_sql = pd.read_sql_query('SELECT * FROM finance_q1_changes', db)
finance_pd = finance_analyze(finance)
finance_keys = ['corp_code','bsns_year','reprt_code','fs_div','sj_div','account_nm']
finance_metrics = ['source_rows','current_amount','previous_amount','change_amount','change_pct']
finance_checks = compare_results(finance_sql, finance_pd, finance_keys, finance_metrics, atol=1e-8)
assert all(x['status'] == 'PASS' for x in finance_checks), finance_checks
selected = finance_sql.query("corp_code == '00126380' and bsns_year == '2024' and fs_div == 'CFS'")
print('Q1 재무 선택 행', len(finance_sql), 'SQL/Pandas 검사', len(finance_checks))
display(selected[['account_nm','current_amount','previous_amount','change_pct','source_rows']].sort_values('account_nm'))

Q1 재무 선택 행 16 SQL/Pandas 검사 14


,account_nm,current_amount,previous_amount,change_pct,source_rows
9,당기순이익(손실),34451351000000,15487100000000,122.451918,2
10,매출액,300870903000000,258935494000000,16.195311,1
11,영업이익,32725961000000,6566976000000,398.341413,1
8,자산총계,514531948000000,455905980000000,12.859223,1


In [3]:
disclosure_sql = pd.read_sql_query('SELECT * FROM disclosure_q2_by_type', db)
monthly_sql = pd.read_sql_query('SELECT * FROM disclosure_q2_monthly', db)
disclosure_pd, monthly_pd = disclosure_analyze(disclosures)
disclosure_checks = compare_results(disclosure_sql, disclosure_pd,
    ['corp_code','period','report_type'], ['count','correction_count'])
assert all(x['status'] == 'PASS' for x in disclosure_checks), disclosure_checks
month_pair = monthly_sql.merge(monthly_pd, on=['corp_code','period'],
    suffixes=('_sql','_pd'), validate='one_to_one')
assert len(month_pair) == len(monthly_sql) == len(monthly_pd) == 24
for metric in ['count','correction_count','candidate']:
    assert (month_pair[metric+'_sql'] == month_pair[metric+'_pd']).all(), metric
candidates = monthly_sql.query("period >= '2024-01' and period <= '2024-12' and candidate == 1")
print('Q2 공시 유형 그룹', len(disclosure_sql), '월', len(monthly_sql),
      '정정·첨부추가', int(disclosure_sql.correction_count.sum()),
      'SQL/Pandas 검사', len(disclosure_checks))
display(candidates[['period','count','previous_count','change_count','change_pct']])

Q2 공시 유형 그룹 197 월 24 정정·첨부추가 30 SQL/Pandas 검사 11


,period,count,previous_count,change_count,change_pct
14,2024-03,23,6.0,17.0,283.333333
17,2024-06,30,5.0,25.0,500.000000
20,2024-09,31,4.0,27.0,675.000000
21,2024-10,37,15.0,22.0,146.666667
22,2024-11,32,16.0,16.0,100.000000
23,2024-12,43,27.0,16.0,59.259259


## Q3 해석과 한계
2024년 연결 재무 4개 지표의 전기 대비 변화와 2024년 공시 월별 50%/10건 기준 후보를 함께 제시한다. 재무는 연간, 공시는 월간이므로 둘의 발생 시점을 인과로 연결하지 않는다. 동일 Open DART 원본을 두 방법으로 계산한 일치이며 원본의 진위나 기업 위험을 증명하지 않는다. 자세한 수치·선정 이유·AI 검증 기록은 result_report.md와 ai_log.md를 참조한다.

In [4]:
db.close()
print('통합 검증 완료')

통합 검증 완료
